# Agent Skills Generation via GEPA `optimize_anything`

This notebook generates modular, file-based skill definitions for the At-Bat Assistant agent using
GEPA's [`optimize_anything`](https://gepa-ai.github.io/gepa/blog/2026/02/18/introducing-optimize-anything/)
API in **Generalization mode** (seedless), then writes them to a Unity Catalog Volume.

## What is `optimize_anything`?

`optimize_anything` is a declarative API from the [GEPA project](https://gepa-ai.github.io/gepa/)
that optimizes any artifact representable as text: code, prompts, agent architectures, configurations,
and -- as we use it here -- **agent skills**. You declare what to optimize and how to measure it; the
system handles the search. The core loop works as follows:

1. An **evaluator** scores a candidate text artifact and returns diagnostic feedback
   (called *Actionable Side Information*, or ASI) alongside the score
2. A **reflection LM** reads the ASI, diagnoses weaknesses, and proposes a targeted improvement
3. A **Pareto-efficient search** preserves candidates that excel on different evaluation aspects,
   preventing averaging from discarding specialized strengths
4. Steps 1-3 repeat until the budget is exhausted, yielding a Pareto frontier of optimized artifacts

The API supports three modes. This notebook uses **Generalization mode**, where a `dataset` and
`valset` are provided so the optimized artifact must generalize to unseen examples -- the same
paradigm used to [learn repository-specific coding agent skills](https://gepa-ai.github.io/gepa/blog/2026/02/18/automatically-learning-skills-for-coding-agents/)
that demonstrates how repository-specific coding-agent skills can improve pass rate while reducing resolution time.

## How We Apply It Here

We treat agent skills as a **text artifact to be iteratively optimized from scratch**:

1. All four evidence sources (UC Functions, System Prompt, Aligned Judge, Evaluated Traces) are
   passed as `background` context to `optimize_anything`
2. In **seedless mode** (`seed_candidate=None`), the reflection LM bootstraps the first skill set
   from the objective and background alone -- no hand-authored seed is required
3. An **evaluator** scores each candidate skill set against held-out traces using the aligned judge
4. The optimization loop (reflection, mutation, Pareto selection) iteratively improves the skills
   so they generalize across the full trace distribution

Because the optimization loop directly measures skill quality against the aligned judge, the
resulting skills are calibrated to the actual SME-defined quality bar established during judge
alignment.

## Architecture

```
                                              optimize_anything(
UC Functions  ─────┐                            seed_candidate = None,
System Prompt ─────┤                            evaluator = judge_scorer,
Judge Guidelines ──┼── background ──────────>   dataset = train_traces,
Trace Analysis ────┘                            valset = val_traces,
                                                objective = "Generate modular skills...",
                                                background = <all four sources>,
                                              )
                                                        |
                                                        v
                                               Optimized Skill MD Files
                                                    --> UC Volume
```

## Prerequisites

1. Run `00_setup.ipynb` to generate `config/atbat_assistant.json`
2. Run `04-Evaluation.ipynb` to produce evaluation traces tagged with `eval: complete`
3. Complete labeling sessions in the Review App (SME feedback)
4. Run `05-JudgeAlignment.ipynb` to produce the aligned judge

In [ ]:
%pip install -U -qqqq backoff databricks-openai uv databricks-agents "mlflow>=3.9" dspy databricks-mcp langgraph-checkpoint-postgres "psycopg[binary,pool]" databricks-langchain langgraph "gepa==0.1.1"
dbutils.library.restartPython()

## Load Configuration

In [ ]:
import json
import os
from pathlib import Path
import mlflow

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())

CATALOG = CONFIG["workspace"]["catalog"]
SCHEMA = CONFIG["workspace"]["schema"]

EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)

PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
REFLECTION_MODEL = os.getenv(
    "SKILLS_REFLECTION_MODEL",
    CONFIG.get("skills", {}).get("reflection_model") or "databricks:/gpt-5-4-external",
)
ALIGNED_JUDGE_NAME = CONFIG["judges"]["aligned_judge_name"]
LLM_ENDPOINT_NAME = CONFIG["llm"]["endpoint_name"]

SKILLS_VOLUME_NAME = CONFIG["skills"]["gepa_volume_name"]
SKILLS_VOLUME_PATH = CONFIG["skills"]["gepa_volume_path"]
UC_TOOL_NAMES = CONFIG["tools"]["uc_tool_names"]

print(f"Catalog/Schema: {CATALOG}.{SCHEMA}")
print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Prompt: {PROMPT_NAME}")
print(f"Reflection Model: {REFLECTION_MODEL}")
print(f"Aligned Judge: {ALIGNED_JUDGE_NAME}")
print(f"Skills Volume (GEPA): {SKILLS_VOLUME_PATH}")
print(f"UC Tools: {len(UC_TOOL_NAMES)} functions")

## Source 1: UC Function Metadata

Load the UC tools using `UCFunctionToolkit` -- same path as the agent and notebook `07`.

In [ ]:
from databricks_langchain import UCFunctionToolkit

print(f"Loading {len(UC_TOOL_NAMES)} UC tools via UCFunctionToolkit...")
uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
uc_tools = uc_toolkit.tools
print(f"Loaded {len(uc_tools)} tools\n")

uc_function_short_names = [name.split(".")[-1] for name in UC_TOOL_NAMES]

uc_functions_summary = []
uc_functions_text = ""

for tool in uc_tools:
    params_text = ""
    if hasattr(tool, "args_schema") and tool.args_schema:
        try:
            schema = tool.args_schema.schema() if hasattr(tool.args_schema, "schema") else {}
        except Exception:
            schema = tool.args_schema.model_json_schema() if hasattr(tool.args_schema, "model_json_schema") else {}
        properties = schema.get("properties", {})
        required_params = schema.get("required", [])
        param_lines = []
        for pname, pinfo in properties.items():
            ptype = pinfo.get("type", "unknown")
            pdesc = pinfo.get("description", "")
            req_str = " (required)" if pname in required_params else ""
            line = f"  - {pname} ({ptype}{req_str})"
            if pdesc:
                line += f": {pdesc}"
            param_lines.append(line)
        params_text = "\n".join(param_lines)

    summary = {
        "name": tool.name,
        "description": tool.description or "(no description)",
        "parameters": params_text,
    }
    uc_functions_summary.append(summary)

    uc_functions_text += f"\n### {tool.name}\n"
    uc_functions_text += f"Description: {summary['description']}\n"
    if params_text:
        uc_functions_text += f"Parameters:\n{params_text}\n"

print(f"--- UC Functions Summary ({len(uc_functions_summary)} tools) ---")
print(uc_functions_text[:3000] + "..." if len(uc_functions_text) > 3000 else uc_functions_text)

## Source 2: System Prompt

In [ ]:
prompt_obj = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
system_prompt_text = prompt_obj.template

print(f"Loaded system prompt: {PROMPT_NAME} (version {prompt_obj.version})")
print(f"Prompt length: {len(system_prompt_text)} characters")
print(f"\n--- System Prompt Preview ---")
print(system_prompt_text[:1500] + "..." if len(system_prompt_text) > 1500 else system_prompt_text)

## Source 3: Aligned Judge

Load the aligned judge. We need both its guidelines (for the seed generation prompt) and
the scorer itself (for the evaluator function).

In [ ]:
from mlflow.genai.scorers import get_scorer

aligned_judge = get_scorer(name=ALIGNED_JUDGE_NAME)
print(f"Loaded aligned judge: {aligned_judge.name}")

print("Triggering episodic memory initialization (dummy score call)...")
try:
    _dummy_result = aligned_judge(
        inputs={"input": [{"role": "user", "content": "How does Max Scherzer pitch to lefties?"}]},
        outputs={"response": "Scherzer relies on his fastball-slider combination against left-handed hitters."},
    )
    print(f"  Dummy score returned: {_dummy_result}")
except Exception as _e:
    print(f"  Dummy score call raised {type(_e).__name__} (expected if model endpoint is unavailable)")
    print(f"  Episodic memory should still be initialized from the attempt.")

semantic_guidelines = []
if hasattr(aligned_judge, "_semantic_memory") and aligned_judge._semantic_memory:
    for i, guideline in enumerate(aligned_judge._semantic_memory, 1):
        guideline_entry = {
            "index": i,
            "text": guideline.guideline_text,
            "source_traces": guideline.source_trace_ids[:3] if guideline.source_trace_ids else [],
        }
        semantic_guidelines.append(guideline_entry)
    print(f"Semantic memory: {len(semantic_guidelines)} distilled guidelines")
else:
    print("WARNING: No semantic memory found on aligned judge. Run 05-JudgeAlignment.ipynb first.")

episodic_examples = []
if hasattr(aligned_judge, "_episodic_memory") and aligned_judge._episodic_memory:
    episodic_examples = aligned_judge._episodic_memory
    print(f"Episodic memory: {len(episodic_examples)} stored examples")
else:
    print("WARNING: No episodic memory found on aligned judge after initialization attempt.")

aligned_instructions = aligned_judge.instructions if hasattr(aligned_judge, "instructions") else ""
print(f"Aligned instructions length: {len(aligned_instructions)} characters")

judge_guidelines_text = ""
for g in semantic_guidelines:
    judge_guidelines_text += f"\n{g['index']}. {g['text']}\n"

episodic_text = ""
if episodic_examples:
    episodic_text = "\n### Episodic Memory (Representative Examples)\n"
    for i, example in enumerate(episodic_examples[:10], 1):
        episodic_text += f"\nExample {i}:\n"
        if hasattr(example, "inputs"):
            episodic_text += f"  Input: {str(example.inputs)[:200]}\n"
        if hasattr(example, "outputs"):
            episodic_text += f"  Output: {str(example.outputs)[:200]}\n"
        if hasattr(example, "score"):
            episodic_text += f"  Score: {example.score}\n"
        if hasattr(example, "rationale"):
            episodic_text += f"  Rationale: {str(example.rationale)[:200]}\n"

print(f"\n--- Aligned Judge Guidelines ---")
print(judge_guidelines_text[:2000] + "..." if len(judge_guidelines_text) > 2000 else judge_guidelines_text)
if episodic_text:
    print(f"\n--- Episodic Memory ({len(episodic_examples)} examples) ---")
    print(episodic_text[:2000] + "..." if len(episodic_text) > 2000 else episodic_text)

## Source 4: Evaluated Traces

Pull traces tagged with `eval: complete`. These will be split into train/val sets for the
`optimize_anything` generalization mode.

In [ ]:
eval_traces_list = mlflow.search_traces(
    locations=[EXPERIMENT_ID],
    filter_string="tag.eval = 'complete'",
    return_type="list",
)

print(f"Loaded {len(eval_traces_list)} traces with tag.eval = 'complete'")

trace_summaries = []
tool_call_counter = {}

for trace in eval_traces_list:
    trace_info = trace.info
    trace_data = trace.data

    user_query = ""
    try:
        request = trace_data.request
        if isinstance(request, str):
            request = json.loads(request)
        if isinstance(request, dict):
            inputs = request.get("input", request.get("inputs", []))
            if isinstance(inputs, list):
                for msg in inputs:
                    if isinstance(msg, dict) and msg.get("role") == "user":
                        user_query = msg.get("content", "")
                        break
            elif isinstance(inputs, dict):
                input_list = inputs.get("input", [])
                for msg in input_list:
                    if isinstance(msg, dict) and msg.get("role") == "user":
                        user_query = msg.get("content", "")
                        break
    except Exception:
        pass

    agent_response = ""
    try:
        response = trace_data.response
        if isinstance(response, str):
            response = json.loads(response)
        if isinstance(response, dict):
            output = response.get("output", [])
            if isinstance(output, list):
                for item in output:
                    if isinstance(item, dict):
                        content = item.get("content", [])
                        if isinstance(content, list):
                            for c in content:
                                if isinstance(c, dict) and c.get("type") == "output_text":
                                    agent_response = c.get("text", "")
                                    break
                        if agent_response:
                            break
    except Exception:
        pass

    tool_calls_in_trace = []
    used_genie = False
    try:
        for span in trace_data.spans:
            span_name = span.name if hasattr(span, "name") else ""
            span_type = span.span_type if hasattr(span, "span_type") else ""

            if span_type == "TOOL" or any(fn in span_name for fn in uc_function_short_names):
                tool_name = span_name.split(" (")[0] if " (" in span_name else span_name
                tool_calls_in_trace.append(tool_name)
                tool_call_counter[tool_name] = tool_call_counter.get(tool_name, 0) + 1
            elif "genie_fallback" in span_name.lower() or "genie" in span_name.lower():
                tool_name = "genie_space_query"
                tool_calls_in_trace.append(tool_name)
                tool_call_counter[tool_name] = tool_call_counter.get(tool_name, 0) + 1
                used_genie = True
            elif "sufficiency_eval" in span_name.lower():
                tool_name = span_name.split(" (")[0] if " (" in span_name else span_name
                tool_calls_in_trace.append(tool_name)
                tool_call_counter[tool_name] = tool_call_counter.get(tool_name, 0) + 1
    except Exception:
        pass

    assessments = []
    raw_assessments = None
    try:
        for location in [trace, trace_info, trace_data]:
            if hasattr(location, "assessments") and location.assessments:
                raw_assessments = location.assessments
                break

        if raw_assessments:
            for assessment in raw_assessments:
                a_info = {"name": getattr(assessment, "name", "unknown")}
                score = None
                for score_attr in ["numeric_value", "value", "score", "feedback_value"]:
                    val = getattr(assessment, score_attr, None)
                    if val is not None:
                        try:
                            score = float(val)
                            break
                        except (ValueError, TypeError):
                            pass
                if score is not None:
                    a_info["score"] = score
                for rationale_attr in ["rationale", "comment", "feedback", "reason"]:
                    val = getattr(assessment, rationale_attr, None)
                    if val:
                        a_info["rationale"] = str(val)[:300]
                        break
                source = getattr(assessment, "source", None)
                if source:
                    if isinstance(source, dict):
                        a_info["source_type"] = source.get("source_type", "unknown")
                    elif hasattr(source, "source_type"):
                        a_info["source_type"] = str(source.source_type)
                assessments.append(a_info)
    except Exception:
        pass

    trace_summaries.append({
        "trace_id": trace_info.trace_id,
        "user_query": user_query[:500],
        "response_preview": agent_response[:500],
        "tool_calls": tool_calls_in_trace,
        "num_tool_calls": len(tool_calls_in_trace),
        "used_genie": used_genie,
        "assessments": assessments,
    })

print(f"\nExtracted summaries from {len(trace_summaries)} traces")
print(f"\n--- Tool Usage Frequency ---")
for tool, count in sorted(tool_call_counter.items(), key=lambda x: -x[1]):
    print(f"  {tool}: {count} calls")

## Build Trace Analysis Text

Compact trace analysis for the seed generation prompt.

In [ ]:
from collections import Counter

tool_sequences = [tuple(sorted(set(ts["tool_calls"]))) for ts in trace_summaries if ts["tool_calls"]]
sequence_counts = Counter(tool_sequences)

genie_traces = [ts for ts in trace_summaries if ts.get("used_genie", False)]
genie_pct = (len(genie_traces) / len(trace_summaries) * 100) if trace_summaries else 0

traces_analysis_text = f"Total evaluated traces: {len(trace_summaries)}\n"
traces_analysis_text += f"Traces that used Genie fallback: {len(genie_traces)} ({genie_pct:.0f}%)\n"
traces_analysis_text += f"\n### Tool Usage Frequency\n"
for tool, count in sorted(tool_call_counter.items(), key=lambda x: -x[1]):
    traces_analysis_text += f"- {tool}: {count} calls across all traces\n"

traces_analysis_text += f"\n### Common Tool Co-occurrence Patterns\n"
for seq, count in sequence_counts.most_common(15):
    traces_analysis_text += f"- {' + '.join(seq)}: {count} traces\n"

traces_analysis_text += f"\n### Representative Queries by Pattern\n"
pattern_examples = {}
for ts in trace_summaries:
    pattern_key = tuple(sorted(set(ts["tool_calls"])))
    if pattern_key not in pattern_examples:
        pattern_examples[pattern_key] = []
    if len(pattern_examples[pattern_key]) < 2:
        pattern_examples[pattern_key].append(ts["user_query"][:200])

for pattern, examples in list(pattern_examples.items())[:10]:
    traces_analysis_text += f"\nPattern: {' + '.join(pattern) if pattern else '(no tools)'}\n"
    for ex in examples:
        traces_analysis_text += f'  - "{ex}"\n'

all_scores = []
for ts in trace_summaries:
    for a in ts["assessments"]:
        if "score" in a:
            all_scores.append(a["score"])

if all_scores:
    traces_analysis_text += f"\n### Assessment Score Distribution\n"
    score_counts = Counter(all_scores)
    for score in sorted(score_counts.keys()):
        traces_analysis_text += f"- Score {score}: {score_counts[score]} assessments\n"
    traces_analysis_text += f"- Mean score: {sum(all_scores) / len(all_scores):.2f}\n"

scored_traces = []
for ts in trace_summaries:
    for a in ts["assessments"]:
        if "score" in a and "rationale" in a:
            scored_traces.append({
                "query": ts["user_query"][:150],
                "score": a["score"],
                "rationale": a["rationale"][:200],
                "judge_name": a["name"],
            })

scored_traces.sort(key=lambda x: x["score"], reverse=True)
if scored_traces:
    traces_analysis_text += "\n### Sample Assessment Rationales\n"
    traces_analysis_text += "\nHigh-scoring examples:\n"
    for st in scored_traces[:3]:
        traces_analysis_text += f'  - Score {st["score"]} ({st["judge_name"]}): "{st["query"][:100]}"\n'
        traces_analysis_text += f'    Rationale: {st["rationale"]}\n'
    traces_analysis_text += "\nLow-scoring examples:\n"
    for st in scored_traces[-3:]:
        traces_analysis_text += f'  - Score {st["score"]} ({st["judge_name"]}): "{st["query"][:100]}"\n'
        traces_analysis_text += f'    Rationale: {st["rationale"]}\n'

if genie_traces:
    traces_analysis_text += f"\n### Genie Fallback Queries\n"
    for gt in genie_traces[:5]:
        traces_analysis_text += f'  - "{gt["user_query"][:200]}"\n'

print(f"Trace analysis: {len(traces_analysis_text):,} chars")
print(traces_analysis_text[:2000] + "..." if len(traces_analysis_text) > 2000 else traces_analysis_text)

## Build Background Context for `optimize_anything`

Assemble all four evidence sources into a single `background` string that the reflection LM
will use to bootstrap and iterate on skill candidates. This is the domain knowledge that
guides the optimization. This feeds into the iterative search loop rather than a
single-shot generation call.

In [ ]:
BACKGROUND = f"""You are optimizing modular skill files for a baseball hitting analysis AI agent.

Each skill is a folder containing exactly three REQUIRED files:

skill-name/
├── SKILL.md   (main instructions: YAML frontmatter, Tools, Workflow, Quality expectations, Response format)
├── GOTCHA.md  (at least 3 specific failure modes with strong language: NEVER, MUST, CRITICAL)
└── EXAMPLES.md (at least 2 worked examples with user query, tool call sequence, expected response)

All three files are MANDATORY for every skill. Skills missing any file will score poorly.
Skills are loaded on-demand at runtime to guide tool selection and response quality.

Below are the four evidence sources that define this agent's capabilities and quality bar.
Use ALL of them when generating and refining skills.

=== SOURCE 1: UC FUNCTIONS (what the agent can do) ===
These are the typed SQL functions available via UCFunctionToolkit:
{uc_functions_text}

=== SOURCE 2: SYSTEM PROMPT (how the agent is instructed to behave) ===
{system_prompt_text}

=== SOURCE 3: ALIGNED JUDGE (what good output looks like, from SME feedback) ===
Distilled guidelines from human expert feedback via MemAlign:
{judge_guidelines_text}

{episodic_text}

Full aligned judge instructions (truncated):
{aligned_instructions}

=== SOURCE 4: EVALUATED TRACES (what the agent actually does in practice) ===
Analysis of traces with tag.eval = 'complete' (human-reviewed, SME-scored):
{traces_analysis_text}

=== SOURCE 5: HELD-OUT SKILLS REGRESSION LESSONS (must be fixed) ===
A GPT 5.4 held-out comparison scored the base agent at 2.20 and the previous skills agent at 1.75. The main regressions were not missing skill structure. They were behavior regressions:
- CRITICAL: If a UC function returns non-empty rows, the agent MUST synthesize those rows and MUST NOT say the specific data is unavailable.
- CRITICAL: Do not let a generic skill override filtered UC evidence. Count, handedness, runner state, season, and matchup tool outputs are authoritative when rows are present.
- CRITICAL: Use Genie only when UC tools are empty, insufficient, or the user asks for an aggregate not covered by UC functions. Do not call Genie just to second-guess answered UC outputs.
- For matchup summaries, do not confuse expected stats or quality-of-contact fields with actual hits or outcomes. Preserve sample size and separate actual results from estimated metrics.
- For arsenal-only questions, list only returned pitch types unless tendency tools were called. The deterministic pitch inventory behavior improved scores and should be preserved.
- Data limitation handling is narrow: use it only for empty rows, tool failures, truncated outputs, or missing requested fields. It must not trigger when relevant rows were returned.

=== SKILL FILE FORMAT ===

Each skill is a folder with exactly 3 REQUIRED files. The candidate JSON array must
contain all 3 files for every skill. Missing files will score poorly.

--- FILE 1: skill-name/SKILL.md ---

---
name: lowercase-with-hyphens (max 64 chars)
description: >
  Third-person description of what this skill does and when to use it.
  Include specific trigger words and query patterns. Max 1024 characters.
---

# [Human-Readable Skill Title]

## Tools
[List each UC function this skill uses. State its role in ONE line.]
[If the skill uses the Genie fallback, state that here.]
[Every skill MUST reference at least 2 tools. Single-tool wrapper skills are not useful.]

## Workflow
[Numbered steps for how the agent should execute this skill.]
[MUST include conditional branching (IF/WHEN/Scenario) for different user intents.]
[MUST include fallback paths for when primary tools return empty data.]
[Include parallel call opportunities where tools are independent.]

## Quality expectations
[What makes a response score 4-5 vs. 1-2 for this specific skill.]
[Derived from the aligned judge guidelines and SME feedback above.]

## Response format
[Expected output structure. Reference the system prompt formatting rules.]

## Before Responding (Mandatory)
[Imperative checklist the agent MUST verify before returning output.]
[Each item uses a checkbox: "- [ ] Every percentage has a raw count (e.g., 45% (90/200))"]
[Derived from Quality expectations above. Minimum 3 checks.]
[Frame as active verification, not passive expectations.]

--- FILE 2: skill-name/GOTCHA.md ---

## Gotchas
[At least 3 specific failure modes to avoid, derived from low-scoring traces.]
[Use strong language: "NEVER", "MUST", "CRITICAL", "DO NOT".]
[Include data quality traps: scaled values, missing sample sizes, implausible numbers.]

--- FILE 3: skill-name/EXAMPLES.md ---

## Examples
[At least 2 worked examples drawn from the evaluated traces.]
[Each example MUST include: the user query, the exact tool call sequence, and a
brief description of the expected response.]

=== CONSTRAINTS ===
1. Group tools that are FREQUENTLY CALLED TOGETHER in the traces (use co-occurrence patterns).
2. Every skill must use at least 2 different tools. Do NOT create single-tool wrapper skills
   (e.g., a "player-identification" skill that only calls lookup_player_by_name is wasteful).
   Instead, fold the lookup step into the skills that need it.
3. Include a skill for Genie fallback (when UC functions cannot answer).
4. Target 7-10 skills total. Fewer, richer skills are better than many thin ones.
5. The candidate MUST be a valid JSON array of {{"filename": "...", "content": "..."}} objects.
6. CRITICAL FILENAME FORMAT: Every filename MUST contain a slash. The pattern is:
     "skill-name/SKILL.md"
     "skill-name/GOTCHA.md"
     "skill-name/EXAMPLES.md"
   Do NOT create flat files like "skill-name.md". Every skill has exactly 3 entries in the
   JSON array, all under the same folder prefix. Example for a 5-skill candidate:
   15 total entries (5 skills x 3 files each).
7. Each skill must be fully self-contained and independently loadable.
8. Every Workflow section MUST contain at least one conditional branch (IF/WHEN/Scenario)
   and at least one fallback path for empty or insufficient data.
9. Every GOTCHA.md MUST document at least 3 specific failure modes.
10. Every EXAMPLES.md MUST include at least 2 worked examples with tool call sequences.
11. Every SKILL.md MUST include a "## Before Responding (Mandatory)" section with at least
    3 checkbox items the agent verifies before returning its response. Frame as active
    verification (e.g., "- [ ] Every percentage includes a raw count"), not passive expectations.
"""



print(f"Background context assembled: {len(BACKGROUND):,} characters")
print(f"  UC functions: {len(uc_functions_text):,} chars")
print(f"  System prompt: {len(system_prompt_text):,} chars")
print(f"  Judge guidelines: {len(judge_guidelines_text):,} chars")
print(f"  Trace analysis: {len(traces_analysis_text):,} chars")

## Prepare Train/Val Split

Split trace summaries into training and validation sets for generalization mode.
Each example in the dataset is a dict containing the user query, actual tool calls,
the agent response, and any SME assessments -- everything the evaluator needs to
score a candidate skill set against a real interaction.

In [ ]:
import random

random.seed(42)
shuffled = list(trace_summaries)
random.shuffle(shuffled)

split_idx = max(1, int(len(shuffled) * 0.7))
train_traces = shuffled[:split_idx]
val_traces = shuffled[split_idx:]

print(f"Total traces: {len(trace_summaries)}")
print(f"Train: {len(train_traces)}, Val: {len(val_traces)}")
print(f"\nTrain tool distribution:")
train_genie = sum(1 for t in train_traces if t["used_genie"])
print(f"  Genie fallback: {train_genie}/{len(train_traces)}")
print(f"Val tool distribution:")
val_genie = sum(1 for t in val_traces if t["used_genie"])
print(f"  Genie fallback: {val_genie}/{len(val_traces)}")

## Define the Evaluator

The evaluator scores a candidate skill set (a JSON-encoded list of skill files) against
a single trace example. Scores are computed holistically across all skills rather than
matching individual skills to queries.

**Scoring (100% structural, all vary with candidate):**
- Tool coverage (25%) -- do any skills in the set mention the right tools for this trace?
- Tool specificity (20%) -- penalizes kitchen-sink skills that list all tools
- Structural completeness (15%) -- YAML frontmatter, Workflow/Tools/Quality sections
- Skill count (10%) -- rewards producing 7-10 skills
- Workflow depth (10%) -- conditional logic (IF/WHEN/Scenario/FALLBACK) in skills
- Example coverage (10%) -- worked examples with tool sequences
- Gotcha coverage (5%) -- specific failure modes documented
- Checklist coverage (5%) -- pre-response verification items

The aligned judge is not used in this evaluator. Since the traces (and therefore
responses) are fixed, the judge score cannot vary between candidates and provides
no optimization signal. Judge-based evaluation is performed downstream when the
generated skills are loaded into the agent and tested on new queries.

In [ ]:
import gepa.optimize_anything as oa
import re as _re


def _parse_skills_from_candidate(candidate_json: str) -> list[dict]:
    """Parse the candidate JSON string into a list of skill dicts.

    GEPA may generate multi-file skill structures (e.g., skill/SKILL.md,
    skill/GOTCHA.md, skill/EXAMPLES.md). This function merges files that
    share a common folder prefix into a single skill entry with concatenated
    content so that the evaluator sees one skill per domain.

    Flat files (e.g., pitcher-arsenal.md) are also merged with their matching
    folder (pitcher-arsenal/) if both exist, so the evaluator always sees
    the complete skill content including examples and gotchas.
    """
    try:
        match = _re.search(r'\[.*\]', candidate_json, _re.DOTALL)
        raw = json.loads(match.group()) if match else json.loads(candidate_json)
    except (json.JSONDecodeError, TypeError):
        return []

    folders: dict[str, list[dict]] = {}
    flat: list[dict] = []

    for entry in raw:
        fn = entry.get("filename", "")
        if "/" in fn:
            folder = fn.rsplit("/", 1)[0]
            folders.setdefault(folder, []).append(entry)
        else:
            flat.append(entry)

    merged = []
    matched_folders: set[str] = set()

    for entry in flat:
        fn = entry.get("filename", "")
        base = fn.rsplit(".", 1)[0]
        if base in folders:
            sub_entries = sorted(
                folders[base],
                key=lambda e: (0 if "skill" in e["filename"].lower() else 1),
            )
            extra = "\n\n".join(e.get("content", "") for e in sub_entries)
            merged.append({
                "filename": fn,
                "content": entry.get("content", "") + "\n\n" + extra,
            })
            matched_folders.add(base)
        else:
            merged.append(entry)

    for folder, entries in folders.items():
        if folder not in matched_folders:
            entries.sort(key=lambda e: (0 if "skill" in e["filename"].lower() else 1))
            merged_content = "\n\n".join(e.get("content", "") for e in entries)
            merged.append({"filename": f"{folder}.md", "content": merged_content})

    return merged


def _parse_raw_entries(candidate_json: str) -> list[dict]:
    """Parse candidate JSON into raw file entries, preserving subfolder paths.

    Unlike _parse_skills_from_candidate (which merges for evaluation),
    this returns the original entries as-is for writing to the volume.
    """
    try:
        match = _re.search(r'\[.*\]', candidate_json, _re.DOTALL)
        raw = json.loads(match.group()) if match else json.loads(candidate_json)
    except (json.JSONDecodeError, TypeError):
        return []
    return [{"filename": e.get("filename", ""), "content": e.get("content", "")} for e in raw]


def _extract_skill_name(content: str) -> str:
    """Pull the name field from YAML frontmatter."""
    match = _re.search(r'^name:\s*(.+)$', content, _re.MULTILINE)
    return match.group(1).strip() if match else "unknown"


def _extract_skill_description(content: str) -> str:
    """Pull the description from YAML frontmatter."""
    match = _re.search(r'description:\s*>?\s*\n?(.*?)(?=\n\w|\n---)', content, _re.DOTALL)
    if match:
        return " ".join(match.group(1).strip().split())
    return ""


_EXPECTED_SECTIONS = ["tools", "workflow", "quality", "response format", "before responding"]


def _tool_short(name: str) -> str:
    """Normalize a UC tool name to its short function name.

    Handles both dotted (catalog.schema.func) and double-underscore
    (catalog__schema__func) formats.
    """
    return name.replace(".", "__").split("__")[-1].lower()


_ALL_TOOL_SHORTS = {_tool_short(tc) for tc in UC_TOOL_NAMES}


def _score_structural_completeness(skill_content: str) -> tuple[float, list[str]]:
    """Check for YAML frontmatter and expected sections."""
    content_lower = skill_content.lower()
    present = []
    checks = 0
    total = len(_EXPECTED_SECTIONS) + 2  # +2 for frontmatter delimiters

    if content_lower.strip().startswith("---"):
        checks += 1
        present.append("yaml_open")
    if content_lower.count("---") >= 2:
        checks += 1
        present.append("yaml_close")

    for section in _EXPECTED_SECTIONS:
        if f"## {section}" in content_lower or f"# {section}" in content_lower:
            checks += 1
            present.append(section)

    return checks / total, present


def evaluate_skills(candidate: str, example: dict) -> tuple[float, dict]:
    """Evaluate a candidate skill set against a single trace example.

    Scores the full skill set holistically rather than matching individual
    skills to queries. All signals vary with the candidate.
    """
    skills = _parse_skills_from_candidate(candidate)

    if not skills:
        oa.log("PARSE ERROR: Could not parse candidate into skill list")
        return 0.0, {
            "Error": "Failed to parse candidate as JSON array of skills",
            "candidate_preview": candidate[:500],
        }

    query = example["user_query"]
    actual_tools = example["tool_calls"]

    all_skill_content = "\n".join(s.get("content", "") for s in skills)
    all_content_lower = all_skill_content.lower()

    TARGET_MIN, TARGET_MAX = 7, 10
    if TARGET_MIN <= len(skills) <= TARGET_MAX:
        skill_count_score = 1.0
    elif len(skills) < TARGET_MIN:
        skill_count_score = len(skills) / TARGET_MIN
    else:
        skill_count_score = max(0.3, 1.0 - (len(skills) - TARGET_MAX) * 0.1)

    tools_covered = [_tool_short(tc) for tc in actual_tools if _tool_short(tc) in all_content_lower]
    tool_cov = len(tools_covered) / max(len(actual_tools), 1)

    actual_shorts = {_tool_short(tc) for tc in actual_tools}
    mentioned_all = {t for t in _ALL_TOOL_SHORTS if t in all_content_lower}
    tool_spec = len(actual_shorts & mentioned_all) / len(mentioned_all) if mentioned_all else 0.0

    struct_scores_list = []
    struct_all_present = []
    for skill in skills:
        sc, present = _score_structural_completeness(skill.get("content", ""))
        struct_scores_list.append(sc)
        struct_all_present.extend(present)
    struct_score = sum(struct_scores_list) / max(len(struct_scores_list), 1)

    _cond_keywords = ["if ", "scenario ", "when ", "critical", "fallback", "pivot"]
    wf_scores = []
    for skill in skills:
        cl = skill.get("content", "").lower()
        wf_scores.append(min(1.0, sum(1 for kw in _cond_keywords if kw in cl) / 4.0))
    workflow_depth = sum(wf_scores) / max(len(wf_scores), 1)

    _example_markers = ["### example", "### query:", "**user query**:", "**user:**"]
    ex_scores = []
    for skill in skills:
        sl = skill.get("content", "").lower()
        count = sum(sl.count(p) for p in _example_markers)
        ex_scores.append(min(1.0, count / 2.0))
    example_score = sum(ex_scores) / max(len(ex_scores), 1)

    gotcha_markers = ["trap", "gotcha", "error", "failure", "never ", "do not ", "must "]
    gt_scores = []
    for skill in skills:
        cl = skill.get("content", "").lower()
        gt_scores.append(min(1.0, sum(1 for g in gotcha_markers if g in cl) / 3.0))
    gotcha_score = sum(gt_scores) / max(len(gt_scores), 1)

    _checklist_markers = ["- [ ]", "verify", "confirm", "check that", "ensure"]
    cl_scores = []
    for skill in skills:
        sl = skill.get("content", "").lower()
        has_section = "before responding" in sl or "pre-response" in sl
        count = sum(sl.count(m) for m in _checklist_markers)
        cl_scores.append(min(1.0, count / 3.0) if has_section else 0.0)
    checklist_score = sum(cl_scores) / max(len(cl_scores), 1)

    regression_musts = [
        "non-empty rows",
        "must synthesize",
        "must not say",
        "data is unavailable",
        "genie only when",
        "do not call genie",
        "empty rows",
        "sample size",
        "actual results",
        "estimated metrics",
        "arsenal-only",
        "do not infer usage",
    ]
    regression_guardrail_score = sum(1 for phrase in regression_musts if phrase in all_content_lower) / len(regression_musts)

    combined_score = (
        0.20 * tool_cov
        + 0.15 * tool_spec
        + 0.15 * struct_score
        + 0.10 * skill_count_score
        + 0.10 * workflow_depth
        + 0.10 * example_score
        + 0.05 * gotcha_score
        + 0.05 * checklist_score
        + 0.10 * regression_guardrail_score
    )

    sme_score = None
    for a in example.get("assessments", []):
        if "score" in a:
            sme_score = a["score"]
            break

    side_info = {
        "scores": {
            "tool_coverage": tool_cov,
            "tool_specificity": tool_spec,
            "structural_completeness": struct_score,
            "skill_count": skill_count_score,
            "workflow_depth": workflow_depth,
            "example_coverage": example_score,
            "gotcha_coverage": gotcha_score,
            "checklist_coverage": checklist_score,
            "regression_guardrails": regression_guardrail_score,
        },
        "Query": query[:200],
        "Actual_tools": actual_tools,
        "Tools_covered": tools_covered,
        "Structural_sections_found": struct_all_present,
        "Num_skills_total": len(skills),
        "Skill_names": [_extract_skill_name(s.get("content", "")) for s in skills],
    }

    if sme_score is not None:
        side_info["SME_score"] = sme_score

    oa.log(f"Query: {query[:80]}")
    oa.log(f"Skills: {len(skills)} (target: {TARGET_MIN}-{TARGET_MAX})")
    oa.log(f"Score: {combined_score:.3f} [cov={tool_cov:.2f} spec={tool_spec:.2f} "
           f"struct={struct_score:.2f} count={skill_count_score:.2f} wf={workflow_depth:.2f} "
           f"ex={example_score:.2f} gotcha={gotcha_score:.2f} regress={regression_guardrail_score:.2f}]")
    if len(skills) < TARGET_MIN:
        oa.log(f"WARNING: Only {len(skills)} skills produced. Target is {TARGET_MIN}-{TARGET_MAX}.")

    return combined_score, side_info


print("Evaluator defined.")

## Run `optimize_anything` (Generalization Mode, Seedless)

In seedless mode (`seed_candidate=None`), the reflection LM generates the initial skill set
from the `objective` and `background`, then iteratively improves it. The full JSON-serialized
skill set is the candidate artifact. GEPA's reflection loop analyzes ASI (judge rationale,
tool coverage gaps, match failures) and proposes improved skill definitions.

We use the same reflection model as notebook 06 (prompt optimization) since it
needs a large context window for the skill JSON + ASI + background.

In [ ]:
from gepa.optimize_anything import optimize_anything, GEPAConfig, EngineConfig, ReflectionConfig, RefinerConfig
from databricks.sdk import WorkspaceClient
import requests as _requests

MAX_METRIC_CALLS = 10

REFLECTION_ENDPOINT = REFLECTION_MODEL.split('/')[-1]
if REFLECTION_ENDPOINT.startswith('gpt-5-') and not REFLECTION_ENDPOINT.endswith('-external'):
    REFLECTION_ENDPOINT = f"databricks-{REFLECTION_ENDPOINT}"
REFLECTION_ENDPOINTS = [REFLECTION_ENDPOINT]

_workspace_client = WorkspaceClient()
_openai_client = _workspace_client.serving_endpoints.get_open_ai_client()
_host = _workspace_client.config.host.rstrip('/')

def _notebook_headers():
    try:
        token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    except Exception:
        token = _workspace_client.config.authenticate()
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

def _databricks_reflection_lm(prompt):
    messages = [{"role": "user", "content": prompt}] if isinstance(prompt, str) else prompt
    last_error = None
    for endpoint in REFLECTION_ENDPOINTS:
        try:
            if endpoint.endswith('-external'):
                response = _requests.post(
                    f"{_host}/serving-endpoints/{endpoint}/invocations",
                    headers=_notebook_headers(),
                    json={"messages": messages, "temperature": 0.3, "max_tokens": 8000},
                    timeout=120,
                )
                response.raise_for_status()
                return str(response.json()["choices"][0]["message"]["content"])
            response = _openai_client.chat.completions.create(
                model=endpoint,
                messages=messages,
            )
            content = response.choices[0].message.content
            if isinstance(content, list):
                parts = []
                for item in content:
                    if isinstance(item, str):
                        parts.append(item)
                    elif isinstance(item, dict):
                        parts.append(item.get("text") or item.get("content") or json.dumps(item))
                    else:
                        parts.append(str(item))
                return "\n".join(parts)
            return str(content)
        except Exception as e:
            last_error = e
            print(f"Reflection endpoint {endpoint} failed: {type(e).__name__}: {e}")
    raise last_error

config = GEPAConfig(
    engine=EngineConfig(
        max_metric_calls=MAX_METRIC_CALLS,
        cache_evaluation=True,
        display_progress_bar=True,
        parallel=False,
    ),
    reflection=ReflectionConfig(
        reflection_lm=_databricks_reflection_lm,
        reflection_minibatch_size=2,
    ),
    refiner=RefinerConfig(),
)

OBJECTIVE = """Generate and optimize a set of modular skill files for a baseball hitting analysis AI agent.

Each skill is a folder with exactly 3 REQUIRED files:
  - skill-name/SKILL.md (YAML frontmatter + Tools, Workflow, Quality expectations, Response format)
  - skill-name/GOTCHA.md (at least 3 failure modes)
  - skill-name/EXAMPLES.md (at least 2 worked examples with tool sequences)

The candidate MUST be a valid JSON array of {"filename": "...", "content": "..."} objects.
Every filename MUST use the pattern "skill-name/SKILL.md", "skill-name/GOTCHA.md", or
"skill-name/EXAMPLES.md". No flat files (no "skill-name.md"). Every skill needs all 3 files.

Maximize:
1. TOOL COVERAGE: Every tool call pattern observed in the traces must be covered. Each skill
   must reference at least 2 tools. No single-tool wrapper skills.
2. WORKFLOW DEPTH: Every SKILL.md must have conditional branches (IF/WHEN/Scenario) and fallback
   paths for empty data. The agent faces diverse queries; skills must handle edge cases.
3. GOTCHA RICHNESS: Every GOTCHA.md must document at least 3 specific failure modes using strong
   language (NEVER, MUST, CRITICAL). These prevent the most common scoring failures.
4. EXAMPLE QUALITY: Every EXAMPLES.md must include at least 2 worked examples showing the exact
   tool call sequence and expected response shape.
5. QUERY ROUTING: Each user query type should map to exactly one skill via the description.
6. GENERALIZATION: Skills must work for unseen queries, not just memorize training examples.
7. PRE-RESPONSE CHECKLIST: Every SKILL.md must end with a "## Before Responding (Mandatory)"
   section containing at least 3 imperative verification items. These force the agent to
   self-check output quality before returning. Frame as active checks, not passive expectations.
8. REGRESSION PREVENTION: Skills must explicitly prevent the held-out regressions observed in notebook 09:
   non-empty UC rows must be synthesized, generic skill text must not override filtered tool evidence,
   Genie should not be called when UC rows already answer the question, and data limitations should
   only be stated for empty rows, tool failures, truncation, or missing requested fields.

Target 7-10 skills total. Fewer rich skills beat many thin ones.
Each skill name: lowercase-with-hyphens, max 64 chars."""

def _baseline_skill_entries(name, description, tools, workflow_focus):
    tools_text = "\n".join([f"- {tool}: use for deterministic evidence in this workflow." for tool in tools])
    skill_md = f"""---
name: {name}
description: >
  {description}
---

# {name.replace('-', ' ').title()}

## Tools
{tools_text}

## Workflow
1. Identify the player, team, season, count, handedness, and base-state filters in the query.
2. WHEN player identity is ambiguous, call lookup_player_by_name before matchup or tendency tools.
3. Use the most specific deterministic function first, then add arsenal or roster context when useful.
4. IF the specific filter returns no rows, state the limitation and fall back to nearby tendency or repertoire evidence.
5. Synthesize evidence into expected pitch mix, attack zones, and hitter plan.

## Quality expectations
A strong answer uses fetched evidence, includes counts or filters when requested, avoids invented percentages, and gives a concrete hitter recommendation. If a UC function returns non-empty rows, the agent must synthesize those rows and must not say the data is unavailable.

## Response format
Use the production system prompt format: Data collected, Pitcher Approach, Recommendation.

## Before Responding (Mandatory)
- [ ] Every stated tendency is backed by a returned tool result.
- [ ] Non-empty UC rows have been synthesized into the answer instead of being treated as unavailable data.
- [ ] Empty or missing data is explicitly labeled as a limitation.
- [ ] The recommendation is specific enough for a batter or coach to act on.
- [ ] {workflow_focus}
"""
    gotcha_md = f"""## Gotchas
- CRITICAL: NEVER invent pitch percentages, matchup history, or spin/speed values that were not returned by tools.
- CRITICAL: If a UC tool returns non-empty rows, MUST synthesize them and MUST NOT say the specific data is unavailable.
- DO NOT call Genie when UC rows already answer the requested count, handedness, runner-state, season, or matchup filter.
- MUST preserve the requested season, count, handedness, and runner state in the analysis.
- MUST separate actual results from estimated metrics and preserve sample size.
- DO NOT dump raw rows without converting them into hitter-usable attack zones and timing cues.
- NEVER describe normalized Genie values as real mph unless the source explicitly confirms real units.
"""
    examples_md = f"""## Examples

### Example 1
User query: How will Spencer Strider pitch to Juan Soto in 2024?
Tool sequence: lookup_player_by_name, get_batter_pitcher_matchup, pitcher_arsenal_lookup, get_pitcher_tendency_by_count.
Expected response: summarize exact matchup availability, arsenal, likely attack zones, and an actionable Soto plan.

### Example 2
User query: What are Corbin Burnes' tendencies with a runner on second against left-handed batters in 2025?
Tool sequence: lookup_player_by_name, get_pitcher_tendency_with_runners.
Expected response: state the runner/count/handedness filter, summarize returned pitch-location tendencies, and recommend an approach.
"""
    return [
        {"filename": f"{name}/SKILL.md", "content": skill_md},
        {"filename": f"{name}/GOTCHA.md", "content": gotcha_md},
        {"filename": f"{name}/EXAMPLES.md", "content": examples_md},
    ]

BASELINE_SKILLS = []
BASELINE_SKILLS.extend(_baseline_skill_entries(
    "matchup-game-plan",
    "Use for batter versus pitcher matchup questions that need hitter game plans from matchup history, pitcher arsenal, and count tendencies.",
    ["lookup_player_by_name", "get_batter_pitcher_matchup", "pitcher_arsenal_lookup", "get_pitcher_tendency_by_count"],
    "The answer separates exact matchup evidence from broader pitcher tendency evidence.",
))
BASELINE_SKILLS.extend(_baseline_skill_entries(
    "count-and-handedness-tendencies",
    "Use for count-specific pitcher tendency questions by batter handedness, including 0-0, 1-1, 2-2, and two-strike planning.",
    ["lookup_player_by_name", "get_pitcher_tendency_by_count", "pitcher_arsenal_lookup"],
    "The response explicitly names the count and batter handedness used in the tool call.",
))
BASELINE_SKILLS.extend(_baseline_skill_entries(
    "runner-state-tendencies",
    "Use for pitcher tendency questions involving runners on base, scoring position, base occupancy, and handedness filters.",
    ["lookup_player_by_name", "get_pitcher_tendency_with_runners", "pitcher_arsenal_lookup"],
    "The response clearly maps runner-state filters to the returned tendency data.",
))
BASELINE_SKILLS.extend(_baseline_skill_entries(
    "arsenal-and-pitch-mix",
    "Use for questions about what pitch types a pitcher throws, how the arsenal plays, and how a hitter should time or zone those pitches.",
    ["lookup_player_by_name", "pitcher_arsenal_lookup", "get_pitcher_tendency_by_count"],
    "The response distinguishes arsenal inventory from situational pitch selection.",
))
BASELINE_SKILLS.extend(_baseline_skill_entries(
    "team-lineup-matchups",
    "Use for team, roster, lineup, or recommended batter matchup questions that compare multiple hitters against a pitcher.",
    ["lookup_player_by_name", "get_team_batters", "recommend_batter_matchups_by_team", "get_batter_pitcher_matchup"],
    "The response explains why recommended hitters fit the pitcher matchup.",
))
BASELINE_SKILLS.extend(_baseline_skill_entries(
    "genie-fallback-analysis",
    "Use when structured UC functions cannot answer a requested analytic slice and Genie-backed table analysis is needed.",
    ["lookup_player_by_name", "genie_space_query", "pitcher_arsenal_lookup"],
    "The response labels Genie evidence and does not overstate incomplete SQL output.",
))
BASELINE_SKILLS.extend(_baseline_skill_entries(
    "data-limitation-handling",
    "Use when tool outputs are empty, partial, truncated, or insufficient for one part of a multi-part baseball question.",
    ["lookup_player_by_name", "get_batter_pitcher_matchup", "get_pitcher_tendency_by_count", "genie_space_query"],
    "The response separates answered parts from unanswered parts without apologizing or inventing evidence.",
))
BASELINE_SKILLS_JSON = json.dumps(BASELINE_SKILLS)


print(f"Starting optimize_anything (Generalization mode, seeded)")
print(f"  seed_candidate: regression-aware baseline skills")
print(f"  Train: {len(train_traces)} examples, Val: {len(val_traces)} examples")
print(f"  Max metric calls: {MAX_METRIC_CALLS}")
print(f"  Reflection LM: {REFLECTION_MODEL} via endpoints {REFLECTION_ENDPOINTS}")
print(f"  Background context: {len(BACKGROUND):,} chars")
print()

result = optimize_anything(
    seed_candidate=BASELINE_SKILLS_JSON,
    evaluator=evaluate_skills,
    dataset=train_traces,
    valset=val_traces,
    objective=OBJECTIVE,
    background=BACKGROUND,
    config=config,
)

print(f"\nOptimization complete.")
print(f"  Best candidate score: {result.val_aggregate_scores[result.best_idx]:.4f}")
print(f"  Best candidate length: {len(result.best_candidate)} chars")

## Parse Optimized Skills

Extract the optimized skill files from the best candidate returned by `optimize_anything`.

In [ ]:
optimized_candidate = result.best_candidate

# best_candidate is str when seed was str, dict when seed was dict
if isinstance(optimized_candidate, dict):
    optimized_json = list(optimized_candidate.values())[0]
else:
    optimized_json = optimized_candidate

generated_skills = _parse_skills_from_candidate(optimized_json)

# Raw entries preserve subfolder paths for volume writing
raw_skill_entries = _parse_raw_entries(optimized_json)

if not generated_skills:
    raise ValueError(
        "Could not parse optimized candidate as JSON array of skills. "
        f"Raw candidate preview: {optimized_json[:500]}"
    )

print(f"Optimized skill set: {len(generated_skills)} skills")
for skill in generated_skills:
    name = _extract_skill_name(skill.get("content", ""))
    print(f"  - {skill['filename']} (name={name}, {len(skill['content'])} chars)")

## Evaluate Optimized Skills on Validation Set

Score the final optimized skills on the held-out validation traces.

In [ ]:
print("Evaluating optimized skills on full val set...")
opt_json = json.dumps(generated_skills)
opt_scores = []
opt_details = []
for ex in val_traces:
    score, info = evaluate_skills(opt_json, ex)
    opt_scores.append(score)
    opt_details.append(info)

opt_mean = sum(opt_scores) / len(opt_scores) if opt_scores else 0
opt_min = min(opt_scores) if opt_scores else 0
opt_max = max(opt_scores) if opt_scores else 0

print(f"\n{'='*50}")
print(f"GEPA optimize_anything Validation Results")
print(f"  (n={len(val_traces)} held-out traces)")
print(f"{'='*50}")
print(f"  Mean score:  {opt_mean:.4f}")
print(f"  Min score:   {opt_min:.4f}")
print(f"  Max score:   {opt_max:.4f}")
print(f"  Best GEPA candidate score (from optimizer): {result.val_aggregate_scores[result.best_idx]:.4f}")
print(f"{'='*50}")

print(f"\nPer-example breakdown:")
for i, (ex, score, info) in enumerate(zip(val_traces, opt_scores, opt_details)):
    query_preview = ex["user_query"][:80]
    n_skills = info.get("Num_skills_total", 0)
    print(f"  [{i+1:2d}] {score:.3f}  skills={n_skills:2d}  query=\"{query_preview}...\"")

print(f"\nTo re-evaluate after changes, re-run the evaluate_skills() function above.")

## Create UC Volume for Skills

In [ ]:
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{SKILLS_VOLUME_NAME}
    COMMENT 'Agent skill files generated from UC functions, system prompt, aligned judge, and evaluated traces'
""")

print(f"Volume ready: {SKILLS_VOLUME_PATH}")

try:
    existing_files = dbutils.fs.ls(SKILLS_VOLUME_PATH)
    print(f"Existing files in volume: {len(existing_files)}")
    for f in existing_files:
        print(f"  {f.name} ({f.size} bytes)")
except Exception as e:
    print(f"Volume is empty or just created (this is expected on first run)")

## Validate and Write Skills to UC Volume

In [ ]:
import re as _re
from datetime import datetime, timezone


def parse_yaml_frontmatter(content: str) -> dict:
    """Extract name and description from YAML frontmatter in a skill file."""
    match = _re.match(r"^---\s*\n(.*?)\n---", content, _re.DOTALL)
    if not match:
        return {}
    frontmatter = match.group(1)
    result = {}
    for field in ["name", "description"]:
        pattern = rf"^{field}:\s*>?\s*\n?(.*?)(?=\n\w|\Z)"
        field_match = _re.search(pattern, frontmatter, _re.MULTILINE | _re.DOTALL)
        if field_match:
            result[field] = " ".join(field_match.group(1).strip().split())
    return result


def validate_frontmatter(frontmatter: dict, filename: str) -> list[str]:
    """Validate YAML frontmatter against skill rules. Returns list of warnings."""
    warnings = []
    name = frontmatter.get("name", "")
    desc = frontmatter.get("description", "")

    if not name:
        warnings.append(f"{filename}: missing 'name' in frontmatter")
    elif len(name) > 64:
        warnings.append(f"{filename}: name exceeds 64 chars ({len(name)})")
    elif not _re.match(r"^[a-z0-9-]+$", name):
        warnings.append(f"{filename}: name contains invalid chars (must be lowercase, numbers, hyphens)")

    if not desc:
        warnings.append(f"{filename}: missing 'description' in frontmatter")
    elif len(desc) > 1024:
        warnings.append(f"{filename}: description exceeds 1024 chars ({len(desc)})")

    return warnings


print("Validating generated skills...\n")
all_warnings = []
skill_metadata = []

for skill in generated_skills:
    fm = parse_yaml_frontmatter(skill["content"])
    warnings = validate_frontmatter(fm, skill["filename"])
    all_warnings.extend(warnings)
    skill_metadata.append({
        "filename": skill["filename"],
        "name": fm.get("name", ""),
        "description": fm.get("description", "")[:200],
        "size_chars": len(skill["content"]),
    })
    status = "WARN" if warnings else "OK"
    print(f"  [{status}] {skill['filename']:45s} name={fm.get('name', '(missing)')}")
    for w in warnings:
        print(f"         {w}")

if all_warnings:
    print(f"\n{len(all_warnings)} warning(s) found. Skills will still be written.")
else:
    print(f"\nAll {len(generated_skills)} skills passed validation.")

print(f"Cleaning old skill files from {SKILLS_VOLUME_PATH}...")
try:
    existing = dbutils.fs.ls(SKILLS_VOLUME_PATH)
    removed = 0
    for f in existing:
        if f.name.endswith("/"):
            dbutils.fs.rm(f.path, recurse=True)
            removed += 1
        elif f.name.endswith(".md") or f.name == "_manifest.json":
            dbutils.fs.rm(f.path)
            removed += 1
    print(f"  Removed {removed} old file(s)/folder(s)")
except Exception as e:
    print(f"  Could not clean volume (may be empty): {e}")

print(f"\nWriting to {SKILLS_VOLUME_PATH}...")
written_files = []
for entry in raw_skill_entries:
    filename = entry["filename"]
    content = entry["content"]

    if not filename.endswith(".md"):
        filename += ".md"
    filename = filename.replace(" ", "-").lower()

    file_path = f"{SKILLS_VOLUME_PATH}/{filename}"
    parent_dir = os.path.dirname(file_path)
    os.makedirs(parent_dir, exist_ok=True)
    with open(file_path, "w") as f:
        f.write(content)

    written_files.append({"filename": filename, "path": file_path, "size": len(content)})
    print(f"  Wrote: {filename} ({len(content):,} chars)")

manifest = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "generator": "07-AgentSkillsGeneration.ipynb",
    "optimization": {
        "method": "gepa.optimize_anything (Generalization mode, seeded)",
        "max_metric_calls": MAX_METRIC_CALLS,
        "reflection_model": REFLECTION_MODEL,
        "train_traces": len(train_traces),
        "val_traces": len(val_traces),
        "best_candidate_score": round(result.val_aggregate_scores[result.best_idx], 4),
        "val_mean_score": round(opt_mean, 4),
    },
    "sources": {
        "experiment_id": EXPERIMENT_ID,
        "prompt_name": PROMPT_NAME,
        "aligned_judge_name": ALIGNED_JUDGE_NAME,
        "llm_endpoint": LLM_ENDPOINT_NAME,
        "num_traces_analyzed": len(trace_summaries),
        "num_uc_tools": len(uc_functions_summary),
        "num_judge_guidelines": len(semantic_guidelines),
    },
    "skills": skill_metadata,
}

manifest_path = f"{SKILLS_VOLUME_PATH}/_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\n  Wrote: _manifest.json")
print(f"\nTotal: {len(written_files)} skill files + 1 manifest written to {SKILLS_VOLUME_PATH}")

## Lakebase Skipped

Lakebase is disabled in this Free Edition deployment. Skills are written to UC Volumes only.


In [ ]:
print("Lakebase disabled; skills were written to UC Volumes only.")


## Verify Generated Skills

In [ ]:
print(f"=== Files in {SKILLS_VOLUME_PATH} ===\n")
volume_files = dbutils.fs.ls(SKILLS_VOLUME_PATH)
for f in volume_files:
    print(f"  {f.name:45s} {f.size:>8,} bytes")

print(f"\nTotal: {len(volume_files)} files")

In [ ]:
for skill in generated_skills:
    filename = skill["filename"]
    content = skill["content"]
    print(f"\n{'=' * 70}")
    print(f"  {filename}")
    print(f"{'=' * 70}")
    preview_lines = content.split("\n")[:60]
    preview = "\n".join(preview_lines)
    if len(preview) > 2000:
        preview = preview[:2000] + "\n... (truncated)"
    elif len(content.split("\n")) > 60:
        preview += "\n... (truncated)"
    print(preview)

## Next Steps

1. **A/B evaluation**: Run the skill-enhanced agent through the full evaluation pipeline (04-Evaluation)
   and compare judge scores against the base agent on identical queries
2. **Iteration budget**: Increase `max_metric_calls` for a longer search if initial results are promising
3. **Multi-parameter optimization**: Use `dict[str, str]` seed candidates to co-optimize individual
   skills as separate parameters, allowing GEPA to focus improvement on the weakest skills